In [ ]:
# src/training/train.py
from ultralytics import YOLO
import csv
from datetime import datetime
from pathlib import Path

In [ ]:
# src/training/train.py

# --------------------------
# Paths
# --------------------------
DATA_YAML = "/home/tutu/Developement/plasticid/configs/data.yaml"          # your dataset
EXPERIMENTS_DIR = "/home/tutu/Developement/plasticid/experiments"         # where metrics & runs go
MODEL_NAME = "plasticid_v1"                   # folder name for this run
PRETRAINED_MODEL = "yolov8n.pt"              # pretrained YOLOv8n

# --------------------------
# Create experiments folder if missing
# --------------------------
Path(EXPERIMENTS_DIR).mkdir(parents=True, exist_ok=True)
metrics_file = Path(EXPERIMENTS_DIR) / "metrics.csv"

# Create metrics CSV if it doesn't exist
if not metrics_file.exists():
    with open(metrics_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "run_id",
            "mAP50",
            "mAP50_95",
            "HDPE_f1",
            "LDPE_f1",
            "PVC_f1",
            "PET_f1",
            "PP_f1",
            "PS_f1",
            "inference_ms",
            "date"
        ])

# --------------------------
# Load pretrained YOLOv8n model
# --------------------------
model = YOLO(PRETRAINED_MODEL)

# --------------------------
# Train the model
# --------------------------
model.train(
     data=DATA_YAML,
    epochs=1,       # dry run
    imgsz=416,      # smaller images
    batch=1,        # CPU only
    project=EXPERIMENTS_DIR,
    name=MODEL_NAME
)

# --------------------------
# Evaluate the model
# --------------------------
metrics = model.val()  # YOLO returns metrics object

# --------------------------
# Save metrics to CSV
# --------------------------
with open(metrics_file, "a", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        MODEL_NAME,
        metrics.box.map50,   # mAP@0.5
        metrics.box.map,     # mAP@0.5:0.95
        metrics.box.class_result(0)['f1'],  # HDPE
        metrics.box.class_result(1)['f1'],  # LDPE
        metrics.box.class_result(2)['f1'],  # PVC
        metrics.box.class_result(3)['f1'],  # PET
        metrics.box.class_result(4)['f1'],  # PP
        metrics.box.class_result(5)['f1'],  # PS
        0,                       # inference time placeholder
        datetime.today().strftime("%Y-%m-%d")
    ])

print(f"✅ Training complete. Metrics saved to {metrics_file}")
print(f"✅ Trained model saved in {EXPERIMENTS_DIR}/{MODEL_NAME}")

In [ ]:
# src/training/train.py

from ultralytics import YOLO
import csv
from datetime import datetime
from pathlib import Path

# ======================================================
# CONFIGURATION
# ======================================================
DATA_YAML = "/home/tutu/Developement/plasticid/configs/data.yaml"
EXPERIMENTS_DIR = "/home/tutu/Developement/plasticid/experiments"
MODEL_NAME = "plasticid_v1"
PRETRAINED_MODEL = "yolov8n.pt"

EPOCHS = 1        # CPU dry run (increase later)
IMGSZ = 416       # CPU-friendly image size
BATCH = 1         # CPU-safe batch size

# ======================================================
# SETUP
# ======================================================
Path(EXPERIMENTS_DIR).mkdir(parents=True, exist_ok=True)
metrics_file = Path(EXPERIMENTS_DIR) / "metrics.csv"

# Create metrics CSV if missing
if not metrics_file.exists():
    with open(metrics_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "run_id",
            "mAP50",
            "mAP50_95",
            "HDPE_AP50",
            "LDPE_AP50",
            "PVC_AP50",
            "PET_AP50",
            "PP_AP50",
            "PS_AP50",
            "inference_ms",
            "date"
        ])

# ======================================================
# TRAINING
# ======================================================
model = YOLO(PRETRAINED_MODEL)

model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=EXPERIMENTS_DIR,
    name=MODEL_NAME
)

# ======================================================
# VALIDATION
# ======================================================
metrics = model.val()

# Per-class metrics
hdpe = metrics.box.class_result(0)  # (p, r, ap50, ap)
ldpe = metrics.box.class_result(1)
pvc  = metrics.box.class_result(2)
pet  = metrics.box.class_result(3)
pp   = metrics.box.class_result(4)
ps   = metrics.box.class_result(5)

# ======================================================
# METRICS LOGGING
# ======================================================
with open(metrics_file, "a", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        MODEL_NAME,
        metrics.box.map50,   # Overall mAP@0.5
        metrics.box.map,     # Overall mAP@0.5:0.95
        hdpe[2],             # HDPE AP@0.5
        ldpe[2],             # LDPE AP@0.5
        pvc[2],              # PVC AP@0.5
        pet[2],              # PET AP@0.5
        pp[2],               # PP AP@0.5
        ps[2],               # PS AP@0.5
        metrics.speed.get("inference", 0),
        datetime.today().strftime("%Y-%m-%d")
    ])

print("✅ Training completed successfully")
print(f"📊 Metrics saved to: {metrics_file}")
print(f"📦 Model saved in: {EXPERIMENTS_DIR}/{MODEL_NAME}")


In [1]:
from ultralytics import YOLO
from pathlib import Path

# -------------------------
# Paths
# -------------------------
MODEL_PATH = "/home/tutu/pythondev/plasticid/experiments/plasticid_v1/weights/best.pt"
IMAGE_PATH = "/home/tutu/pythondev/plasticid/data/images/test/hdpe (16).png"   # change this
OUTPUT_DIR = "/home/tutu/pythondev/plasticid/inference_output001"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# -------------------------
# Load model (CPU)
# -------------------------
model = YOLO(MODEL_PATH)

# -------------------------
# Run inference
# -------------------------
results = model.predict(
    source=IMAGE_PATH,
    device="cpu",
    conf=0.25,
    save=True,
    project=OUTPUT_DIR,
    name="run1"
)

# -------------------------
# Print detections
# -------------------------
for r in results:
    if r.boxes is None:
        print("No objects detected.")
        continue

    for box in r.boxes:
        cls_id = int(box.cls.item())
        conf = float(box.conf.item())
        class_name = model.names[cls_id]

        print(f"Detected: {class_name} | Confidence: {conf:.2f}")

print("✅ Inference complete")



image 1/1 /home/tutu/pythondev/plasticid/data/images/test/hdpe (16).png: 416x416 2 HDPEs, 174.6ms
Speed: 9.5ms preprocess, 174.6ms inference, 11.0ms postprocess per image at shape (1, 3, 416, 416)
Results saved to /home/tutu/pythondev/plasticid/inference_output001/run1
Detected: HDPE | Confidence: 0.69
Detected: HDPE | Confidence: 0.34
✅ Inference complete


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import csv

# -------------------------
# Paths
# -------------------------
IMAGE_DIR = "/home/tutu/Developement/plasticid/data/images/test"
OUTPUT_DIR = "/home/tutu/Developement/plasticid/batch_inference_output"
CSV_PATH = f"{OUTPUT_DIR}/results.csv"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# -------------------------
# Load model (CPU)
# -------------------------
model = YOLO(MODEL_PATH)

# -------------------------
# Run inference on folder
# -------------------------
results = model.predict(
    source=IMAGE_DIR,
    device="cpu",
    conf=0.25,
    save=True,
    project=OUTPUT_DIR,
    name="images"
)

# -------------------------
# Write CSV
# -------------------------
with open(CSV_PATH, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "image",
        "class",
        "confidence",
        "x1", "y1", "x2", "y2"
    ])

    for r in results:
        image_name = Path(r.path).name

        if r.boxes is None:
            continue

        for box in r.boxes:
            cls_id = int(box.cls.item())
            conf = float(box.conf.item())
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            writer.writerow([
                image_name,
                model.names[cls_id],
                round(conf, 4),
                round(x1, 1),
                round(y1, 1),
                round(x2, 1),
                round(y2, 1),
            ])

print(f"✅ Batch inference complete")
print(f"📊 Results saved to: {CSV_PATH}")
